# 04mc — Conflitos de rótulo entre as 7 classes (e o ajuste sem as linhas conflitantes)

Este notebook **não treina nada**. Ele responde, para as linhas que o notebook 04m selecionou, a pergunta: **quais têm as mesmas features que outra linha do treino com um rótulo DIFERENTE, considerando só as 7 classes do nosso multiclasse?** E prepara o conjunto de ajuste **sem essas linhas**, para a variante de treino do notebook 05mc.

## Por que não usar a marca do notebook 02
O notebook 02 marcou `conflito_rotulo = 1` olhando **as 13 classes** do treino original. Para o nosso multiclasse isso exagera: o `LDAP`, por exemplo, tem 43% das linhas marcadas, mas quase sempre em conflito com `DNS` e `SNMP`, que **saíram**. No conjunto de 7 classes, essas linhas não têm o parceiro conflitante.

| | Marca do 02 (`conflito_rotulo`) | **Aqui (`conflito7`)** |
|---|---|---|
| Rótulos considerados | as 13 classes do treino | **só as 7 classes** (`BENIGN`, `LDAP`, `MSSQL`, `NetBIOS`, `Syn`, `UDP`, `UDPLag`) |
| Linha em conflito quando... | as mesmas features aparecem com 2 ou mais rótulos entre os 13 | as mesmas features aparecem com 2 ou mais rótulos **entre os 7** |

Os rótulos vêm dos grupos de features iguais que o notebook 02 já calculou (`prep_grp_treino`, ligados às linhas por `prep_keep_treino`). Nada é recalculado do zero, e o teste não é tocado.

## Como o conjunto sem conflitos é montado
| Conjunto | O que acontece |
|---|---|
| **Ajuste** | Saem as linhas com `conflito7 = 1`. As demais ficam, **sem sortear de novo** (mesmas linhas do 04m, menos as conflitantes) |
| **Validação** | **Não muda**: as mesmas 59.918 linhas, com os mesmos pesos. Assim o modelo sem conflitos e o modelo normal são medidos **exatamente nas mesmas linhas** |

Os pesos `class_weight="balanced"` serão recalculados pelo próprio RF, a partir do ajuste reduzido (no notebook 05mc).

## O que é criado (nada existente é alterado)
| Objeto | Conteúdo |
|---|---|
| `prep_bal_multi_conf7` | Para cada linha selecionada (ajuste e validação): `conflito13` (a marca do 02), `conflito7` (a daqui), quantos rótulos distintos, quais são as classes parceiras |
| `prep_bal_multi_selecao_sem_conf` e `prep_bal_multi_dados_sem_conf` | A seleção e os dados: ajuste **sem** as linhas conflitantes + validação **inteira** |
| `prep_vw_bal_multi_treino_sem_conf`, `prep_vw_bal_multi_validacao_sem_conf` | As views para o treino, com as mesmas colunas das originais |

## Cuidados de leitura
- **Comparar com o RF normal só vale na mesma validação.** A validação cruzada do ajuste reduzido **não é comparável** com a do ajuste completo (as dobras de avaliação ficam sem as linhas difíceis e parecem melhores).
- **É uma análise de sensibilidade.** Tirar linhas muda a distribuição do treino: a variante só vira candidata por uma regra fixada **antes** do treino (no 05mc).
- Linhas **quase** idênticas (features muito próximas, mas não iguais) **não** são conflitos aqui: a marca só pega features exatamente iguais.

## Antes de rodar
Os notebooks 02, 03m e 04m precisam ter terminado, e as tabelas intermediárias `prep_keep_treino` e `prep_grp_treino` do notebook 02 precisam existir (se tiverem sido apagadas, a célula 1.1 avisa). A conexão é a mesma dos outros (arquivo `.env`). Trava de segurança: só se escreve em `prep_*`.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas ----------
import math
import numpy as np
from datetime import datetime

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte. Aqui, se o servidor estiver sem memória (Code 241) ou a conexão cair, esperamos e refazemos
    # (o comando `antes`, por exemplo um DROP, devolve o estado inicial; tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

# barra de progresso e formatação de tempo (as mesmas dos notebooks de teste)
import gc
def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")

## 1. Parâmetros e levantamento
Confere as tabelas de que o notebook depende e mostra o plano de linhas do 04m.

In [5]:
# ---------- 1.1 Parâmetros e levantamento ----------
SEL_ORIG, DADOS_ORIG, V_TREINO, V_VALID = "prep_bal_multi_selecao", "prep_bal_multi_dados", "prep_vw_bal_multi_treino", "prep_vw_bal_multi_validacao"
CONF_TAB, SEL_SC, DADOS_SC = "prep_bal_multi_conf7", "prep_bal_multi_selecao_sem_conf", "prep_bal_multi_dados_sem_conf"
V_TREINO_SC, V_VALID_SC = "prep_vw_bal_multi_treino_sem_conf", "prep_vw_bal_multi_validacao_sem_conf"
KEEP_TAB, GRP_TAB, TAB_CLASSES = "prep_keep_treino", "prep_grp_treino", "prep_classes_rf_multi"
def colunas_view(view): return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()
N_PEDACOS = globals().get("N_PEDACOS", 12)                      # a seleção é processada em pedaços (por hash do id), para poupar a memória do servidor
faltam = [t for t in ("prep_final_treino", SEL_ORIG, DADOS_ORIG, V_TREINO, V_VALID, TAB_CLASSES, KEEP_TAB, GRP_TAB) if not existe(t)]
assert not faltam, (f"faltam as tabelas {faltam}. Rode antes os notebooks 02, 03m e 04m. As tabelas {KEEP_TAB} e {GRP_TAB} são intermediárias do notebook 02 (seção 4, duplicatas e conflitos): "
                    "se tiverem sido apagadas, é preciso recriá-las rodando essa parte do 02.")
cl = consultar_num(f"SELECT codigo, classe FROM {TAB_CLASSES} ORDER BY codigo"); CLASSES = cl.classe.tolist(); COD = {c: int(k) for c, k in zip(cl.classe, cl.codigo)}
nome_tipo = lambda t: re.sub(r"^DrDoS_", "", str(t))
n_sel = n_linhas(SEL_ORIG); cont = consultar_num(f"SELECT tipo, papel, count() AS n FROM {SEL_ORIG} GROUP BY tipo, papel")
plano = cont.pivot_table(index="tipo", columns="papel", values="n", aggfunc="sum", fill_value=0).astype(int).reindex(CLASSES).rename(columns={"treino": "ajuste", "validacao": "validação"})
print(f"{len(CLASSES)} classes: {CLASSES}\nSeleção do notebook 04m: {n_sel:,} linhas (ajuste + validação)"); display(plano)

7 classes: ['BENIGN', 'LDAP', 'MSSQL', 'NetBIOS', 'Syn', 'UDP', 'UDPLag']
Seleção do notebook 04m: 299,589 linhas (ajuste + validação)


papel,ajuste,validação
tipo,,
BENIGN,40000,10000
LDAP,24079,6020
MSSQL,40000,10000
NetBIOS,15592,3898
Syn,40000,10000
UDP,40000,10000
UDPLag,40000,10000


## 2. Quais rótulos aparecem junto com cada linha
Para cada linha selecionada: pega o grupo de features iguais dela (`prep_keep_treino` → `prep_grp_treino`), lê os **rótulos desse grupo**, tira o prefixo `DrDoS_` e fica só com os que são das **7 classes**. Se sobram **2 ou mais**, a linha está em conflito entre as 7 (`conflito7 = 1`). A `conflito13` é a mesma conta com as 13 classes (equivale à marca do notebook 02, e a célula confere isso). O trabalho é feito em pedaços, para o servidor não passar da memória.

In [6]:
# ---------- 2.1 Rótulos de cada grupo de features iguais ----------
t_ini = time.perf_counter(); partes = []; barra = Barra(N_PEDACOS, "Conflitos entre as classes")
for c in range(N_PEDACOS):
    ids_c = f"SELECT id_linha FROM {SEL_ORIG} WHERE cityHash64(id_linha) % {N_PEDACOS} = {c}"
    d = consultar(f"SELECT toString(s.id_linha) AS id_linha, s.papel AS papel, s.tipo AS tipo, s.y AS y, g.rotulos AS rotulos13 FROM {SEL_ORIG} AS s "
                  f"INNER JOIN (SELECT id_manter, h_feat FROM {KEEP_TAB} WHERE id_manter IN ({ids_c})) AS k ON k.id_manter = s.id_linha "
                  f"INNER JOIN (SELECT h_feat, rotulos FROM {GRP_TAB} WHERE h_feat IN (SELECT h_feat FROM {KEEP_TAB} WHERE id_manter IN ({ids_c}))) AS g ON g.h_feat = k.h_feat "
                  f"WHERE cityHash64(s.id_linha) % {N_PEDACOS} = {c}")
    partes.append(d); barra.avancar(1, f"pedaço {c + 1}/{N_PEDACOS}: {len(d):,} linhas")
barra.fechar()
CF = pd.concat(partes, ignore_index=True); del partes
lista = lambda r: r if isinstance(r, list) else (json.loads(r) if isinstance(r, str) and r.startswith("[") else list(r))
assert len(CF) == n_sel and CF.id_linha.is_unique, f"só {len(CF):,} das {n_sel:,} linhas da seleção foram encontradas em {KEEP_TAB}/{GRP_TAB}"
CF["y"] = CF["y"].astype(int); CF["rot13"] = CF.rotulos13.map(lambda r: sorted({nome_tipo(x) for x in lista(r)})); CF["rot7"] = CF.rot13.map(lambda r: [x for x in r if x in COD])
CF["n13"], CF["n7"] = CF.rot13.map(len), CF.rot7.map(len); CF["conflito13"], CF["conflito7"] = (CF.n13 >= 2).astype(int), (CF.n7 >= 2).astype(int)
CF["parceiros7"] = ["|".join(x for x in r if x != t) for r, t in zip(CF.rot7, CF.tipo)]; CF["combinacao7"] = CF.rot7.map("|".join)
print(f"✔ {len(CF):,} linhas com o grupo de rótulos | tempo {fmt_tempo(time.perf_counter() - t_ini)}")
print(f"   em conflito pela marca do 02 (13 classes): {int(CF.conflito13.sum()):,} ({100 * CF.conflito13.mean():.1f}%) | em conflito entre as 7 classes: {int(CF.conflito7.sum()):,} ({100 * CF.conflito7.mean():.1f}%)")

Conflitos entre as classes: concluído em 10s
✔ 299,589 linhas com o grupo de rótulos | tempo 12s
   em conflito pela marca do 02 (13 classes): 41,668 (13.9%) | em conflito entre as 7 classes: 19,864 (6.6%)


## 3. O que mostram os conflitos
- **Por classe:** quantas linhas do ajuste e da validação estão em conflito pela marca do 02 (13 classes) e **entre as 7 classes**.
- **Quanto da marca do 02 era conflito de verdade** entre as 7.
- **Com quem cada classe se confunde** (pares de classes em conflito) e as combinações de rótulos mais frequentes.
- **O ajuste que sobra** sem as linhas conflitantes, por classe, e os pesos `balanced` que resultariam. Se uma classe perder muita coisa, o aviso aparece.

In [7]:
# ---------- 3.1 Resumo dos conflitos ----------
def por_papel(papel):
    d = CF[CF.papel == papel].groupby("tipo").agg(linhas=("id_linha", "size"), marcadas_02=("conflito13", "sum"), conflito_entre_as_7=("conflito7", "sum")).reindex(CLASSES)
    d["% conflito (7 classes)"] = (100 * d.conflito_entre_as_7 / d.linhas).round(1); d["% marcadas (02)"] = (100 * d.marcadas_02 / d.linhas).round(1); return d
print("== AJUSTE: conflitos por classe =="); aj_r = por_papel("treino"); display(aj_r)
print("== VALIDAÇÃO: conflitos por classe =="); va_r = por_papel("validacao"); display(va_r)
cr = pd.crosstab(CF.conflito13, CF.conflito7).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
print(f"\nDas {int(CF.conflito13.sum()):,} linhas marcadas pelo 02 (13 classes), {int(cr.loc[1, 1]):,} ({100 * cr.loc[1, 1] / max(CF.conflito13.sum(), 1):.1f}%) são conflito de verdade entre as 7 classes; "
      f"{int(cr.loc[1, 0]):,} só tinham como parceiras classes que saíram. (Conflito entre as 7 sem marca do 02: {int(cr.loc[0, 1])}; deve ser 0.)")
pares = pd.DataFrame(0, index=CLASSES, columns=CLASSES)
for t, p in zip(CF.tipo[CF.conflito7 == 1], CF.parceiros7[CF.conflito7 == 1]):
    for x in p.split("|"):
        if x: pares.loc[t, x] += 1
print("\n== Pares em conflito entre as 7 classes: linhas da classe (linha) que têm a classe (coluna) como parceira =="); display(pares)
comb = CF[CF.conflito7 == 1].groupby("combinacao7").size().sort_values(ascending=False).head(12).rename("linhas"); print("Combinações de rótulos mais frequentes (entre as 7):"); display(comb.to_frame())

sem = CF[(CF.papel == "treino") & (CF.conflito7 == 0)]; antes = CF[CF.papel == "treino"].groupby("tipo").size().reindex(CLASSES); depois = sem.groupby("tipo").size().reindex(CLASSES).fillna(0).astype(int)
plano_sc = pd.DataFrame({"ajuste (04m)": antes, "sem conflitos": depois, "ficam %": (100 * depois / antes).round(1)}); n_dep = int(depois.sum()); plano_sc["peso balanced"] = (n_dep / (len(CLASSES) * depois.replace(0, np.nan))).round(3)
print(f"\n== Ajuste sem as linhas em conflito entre as 7 classes: {int(antes.sum()):,} → {n_dep:,} linhas =="); display(plano_sc)
print(f"Razão entre a maior e a menor classe: {int(depois.max()) / max(int(depois.min()), 1):.2f} para 1 (antes: {int(antes.max()) / int(antes.min()):.2f} para 1)")
for c in CLASSES:
    if plano_sc.loc[c, "ficam %"] < 70 or depois[c] < 5000: print(f"⚠ {c} fica com {int(depois[c]):,} linhas ({plano_sc.loc[c, 'ficam %']}% do ajuste): a variante pode ficar pobre nessa classe.")

== AJUSTE: conflitos por classe ==


,linhas,marcadas_02,conflito_entre_as_7,% conflito (7 classes),% marcadas (02)
tipo,,,,,
BENIGN,40000,17,3,0.0,0.0
LDAP,24079,10323,1399,5.8,42.9
MSSQL,40000,3784,2254,5.6,9.5
NetBIOS,15592,8149,6986,44.8,52.3
Syn,40000,647,561,1.4,1.6
UDP,40000,5559,391,1.0,13.9
UDPLag,40000,4843,4302,10.8,12.1


== VALIDAÇÃO: conflitos por classe ==


,linhas,marcadas_02,conflito_entre_as_7,% conflito (7 classes),% marcadas (02)
tipo,,,,,
BENIGN,10000,4,2,0.0,0.0
LDAP,6020,2589,358,5.9,43.0
MSSQL,10000,988,591,5.9,9.9
NetBIOS,3898,1999,1712,43.9,51.3
Syn,10000,178,138,1.4,1.8
UDP,10000,1373,94,0.9,13.7
UDPLag,10000,1215,1073,10.7,12.2



Das 41,668 linhas marcadas pelo 02 (13 classes), 19,864 (47.7%) são conflito de verdade entre as 7 classes; 21,804 só tinham como parceiras classes que saíram. (Conflito entre as 7 sem marca do 02: 0; deve ser 0.)

== Pares em conflito entre as 7 classes: linhas da classe (linha) que têm a classe (coluna) como parceira ==


,BENIGN,LDAP,MSSQL,NetBIOS,Syn,UDP,UDPLag
BENIGN,0,2,0,0,0,2,1
LDAP,2,0,1703,171,17,104,72
MSSQL,0,406,0,2087,7,688,59
NetBIOS,0,171,8541,0,14,1329,103
Syn,0,5,5,2,0,6,694
UDP,0,6,129,55,0,0,357
UDPLag,0,35,118,55,1237,4120,0


Combinações de rótulos mais frequentes (entre as 7):


,linhas
combinacao7,
MSSQL|NetBIOS,9009
UDP|UDPLag,4354
Syn|UDPLag,1923
LDAP|MSSQL,1835
MSSQL|NetBIOS|UDP,1373
MSSQL|UDP,444
LDAP|MSSQL|NetBIOS,251
NetBIOS|UDP,130
MSSQL|NetBIOS|UDP|UDPLag,122



== Ajuste sem as linhas em conflito entre as 7 classes: 239,671 → 223,775 linhas ==


,ajuste (04m),sem conflitos,ficam %,peso balanced
tipo,,,,
BENIGN,40000,39997,100.0,0.799
LDAP,24079,22680,94.2,1.410
MSSQL,40000,37746,94.4,0.847
NetBIOS,15592,8606,55.2,3.715
Syn,40000,39439,98.6,0.811
UDP,40000,39609,99.0,0.807
UDPLag,40000,35698,89.2,0.896


Razão entre a maior e a menor classe: 4.65 para 1 (antes: 2.57 para 1)
⚠ NetBIOS fica com 8,606 linhas (55.2% do ajuste): a variante pode ficar pobre nessa classe.


## 4. Gravação: a tabela de conflitos, a seleção sem conflitos e as views
Grava `prep_bal_multi_conf7` (um registro por linha selecionada), depois cria a seleção e os dados **sem as linhas conflitantes do ajuste** (a validação fica inteira) e as duas views usadas pelo notebook 05mc.

In [8]:
# ---------- 4.1 Gravação ----------
versao_c = datetime.now().strftime("%Y%m%d_%H%M%S")
executar(f"DROP TABLE IF EXISTS `{CONF_TAB}`")
executar(f"CREATE TABLE {CONF_TAB} (id_linha UInt64, papel String, tipo String, y UInt8, conflito13 UInt8, conflito7 UInt8, n_rotulos13 UInt8, n_rotulos7 UInt8, parceiros7 String, combinacao7 String, versao String) ENGINE = MergeTree ORDER BY (papel, id_linha)")
for i in range(0, len(CF), 20_000):
    p = CF.iloc[i:i + 20_000]
    executar_robusto(f"INSERT INTO {CONF_TAB} VALUES " + ", ".join(f"({r.id_linha}, {lit(r.papel)}, {lit(r.tipo)}, {int(r.y)}, {int(r.conflito13)}, {int(r.conflito7)}, {int(r.n13)}, {int(r.n7)}, {lit(r.parceiros7)}, {lit(r.combinacao7)}, {lit(versao_c)})" for r in p.itertuples()))
print(f"✔ {CONF_TAB}: {n_linhas(CONF_TAB):,} linhas")
executar_robusto(f"CREATE TABLE {SEL_SC} ENGINE = MergeTree ORDER BY (papel, id_linha) AS SELECT * FROM {SEL_ORIG} WHERE NOT (papel = 'treino' AND id_linha IN (SELECT id_linha FROM {CONF_TAB} WHERE conflito7 = 1))", antes=f"DROP TABLE IF EXISTS `{SEL_SC}`")
executar_robusto(f"CREATE TABLE {DADOS_SC} ENGINE = MergeTree ORDER BY (papel, id_linha) AS SELECT * FROM {DADOS_ORIG} WHERE id_linha IN (SELECT id_linha FROM {SEL_SC}) SETTINGS max_threads = 2", antes=f"DROP TABLE IF EXISTS `{DADOS_SC}`")
print(f"✔ {SEL_SC}: {n_linhas(SEL_SC):,} linhas | {DADOS_SC}: {n_linhas(DADOS_SC):,} linhas")
cols_modelo = ", ".join(q(c) for c in colunas_view(V_TREINO) if c not in ("id_linha", "y"))
executar(f"DROP VIEW IF EXISTS `{V_TREINO_SC}`"); executar(f"CREATE VIEW {V_TREINO_SC} AS SELECT id_linha, y, {cols_modelo} FROM {DADOS_SC} WHERE papel = 'treino'")
executar(f"DROP VIEW IF EXISTS `{V_VALID_SC}`"); executar(f"CREATE VIEW {V_VALID_SC} AS SELECT id_linha, y, peso, {cols_modelo} FROM {DADOS_SC} WHERE papel = 'validacao'")
print(f"✔ Views: {V_TREINO_SC} e {V_VALID_SC}  | versão {versao_c}")

✔ prep_bal_multi_conf7: 299,589 linhas
✔ prep_bal_multi_selecao_sem_conf: 283,693 linhas | prep_bal_multi_dados_sem_conf: 283,693 linhas
✔ Views: prep_vw_bal_multi_treino_sem_conf e prep_vw_bal_multi_validacao_sem_conf  | versão 20261101_234600


## 5. Validação
Confere, de forma independente: que a tabela de conflitos cobre toda a seleção, que **todo conflito entre as 7 classes é também conflito entre as 13** (e que a nossa conta com 13 classes coincide com a marca do notebook 02), que a seleção sem conflitos é a seleção do 04m **menos só as linhas conflitantes do ajuste**, que a **validação é idêntica** (mesmas linhas e pesos), que os dados copiados são iguais aos originais e que as views têm as colunas certas, sem nulos.

In [9]:
# ---------- 5.1 Validação ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("A tabela de conflitos tem uma linha para cada linha da seleção do 04m", n_linhas(CONF_TAB) == n_sel == int(consultar_num(f"SELECT count() AS n FROM {CONF_TAB} c INNER JOIN {SEL_ORIG} s ON c.id_linha = s.id_linha").n[0]), f"({n_sel:,})")
verifica("Todo conflito entre as 7 classes também é conflito entre as 13", int(((CF.conflito7 == 1) & (CF.conflito13 == 0)).sum()) == 0)
verifica("A classe de cada linha está entre os rótulos do seu grupo de features iguais", all(t in r for t, r in zip(CF.tipo, CF.rot13)))
cols_fin = set(consultar("SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = 'prep_final_treino'")["name"])
if "conflito_rotulo" in cols_fin:
    dif = int(consultar_num(f"SELECT count() AS n FROM {CONF_TAB} c INNER JOIN prep_final_treino f ON f.id_linha = c.id_linha WHERE f.conflito_rotulo != c.conflito13").n[0])
    verifica("A conta com as 13 classes coincide, linha a linha, com a marca conflito_rotulo do notebook 02", dif == 0, f"({dif} divergências)")
else: print("(prep_final_treino não tem a coluna conflito_rotulo: a conferência com a marca do 02 foi omitida)")
n_conf_aj = int(((CF.papel == "treino") & (CF.conflito7 == 1)).sum())
verifica("A seleção sem conflitos = a do 04m menos as linhas conflitantes do ajuste", n_linhas(SEL_SC) == n_sel - n_conf_aj, f"({n_sel:,} − {n_conf_aj:,} = {n_sel - n_conf_aj:,})")
verifica("Nenhuma linha conflitante sobrou no ajuste", int(consultar_num(f"SELECT count() AS n FROM {SEL_SC} s INNER JOIN {CONF_TAB} c ON c.id_linha = s.id_linha WHERE s.papel = 'treino' AND c.conflito7 = 1").n[0]) == 0)
a = consultar_num(f"SELECT count() AS n, sum(peso) AS p, uniqExact(id_linha) AS u FROM {SEL_ORIG} WHERE papel = 'validacao'").iloc[0]; b = consultar_num(f"SELECT count() AS n, sum(peso) AS p, uniqExact(id_linha) AS u FROM {SEL_SC} WHERE papel = 'validacao'").iloc[0]
verifica("A validação ficou idêntica: mesmas linhas e mesmos pesos", int(a.n) == int(b.n) == int(b.u) and abs(float(a.p) - float(b.p)) < 1e-6 * float(a.p) and int(consultar_num(f"SELECT count() AS n FROM {SEL_ORIG} s INNER JOIN {SEL_SC} t ON s.id_linha = t.id_linha WHERE s.papel = 'validacao'").n[0]) == int(a.n), f"({int(a.n):,} linhas)")
verifica("Os dados sem conflitos têm as mesmas linhas da seleção sem conflitos", n_linhas(DADOS_SC) == n_linhas(SEL_SC))
feats = [c for c in colunas_view(V_TREINO) if c not in ("id_linha", "y")]
v1 = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in feats) + f" FROM {DADOS_SC} WHERE cityHash64(id_linha) % 20 = 0 ORDER BY id")
v2 = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in feats) + f" FROM {DADOS_ORIG} WHERE id_linha IN (SELECT id_linha FROM {DADOS_SC} WHERE cityHash64(id_linha) % 20 = 0) ORDER BY id")
verifica("Dados idênticos aos originais (amostra)", len(v1) == len(v2) > 0 and (v1["id"].values == v2["id"].values).all() and np.allclose(v1[["y"] + feats].apply(pd.to_numeric).values.astype(float), v2[["y"] + feats].apply(pd.to_numeric).values.astype(float), rtol=1e-12, atol=1e-12), f"({len(v1):,} linhas)")
for vw, vo in ((V_TREINO_SC, V_TREINO), (V_VALID_SC, V_VALID)):
    c1 = consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(vw)} ORDER BY position")["name"].tolist(); c0 = colunas_view(vo)
    verifica(f"{vw}: mesmas colunas da view original", c1 == c0, f"({len(c1)} colunas)")
    nulos = consultar_num(f"SELECT " + ", ".join(f"countIf(isNull({q(c)})) AS {q(c)}" for c in feats) + f" FROM {vw}").iloc[0]; verifica(f"{vw}: sem nulos nos atributos", int(nulos.sum()) == 0)
verifica("A view de validação tem as mesmas linhas da original", n_linhas(V_VALID_SC) == n_linhas(V_VALID), f"({n_linhas(V_VALID_SC):,})")
cnt = consultar_num(f"SELECT tipo, count() AS n FROM {SEL_SC} WHERE papel = 'treino' GROUP BY tipo"); cnt = dict(zip(cnt.tipo, cnt.n.astype(int)))
verifica("Todas as classes continuam no ajuste sem conflitos, com pelo menos 100 linhas", all(cnt.get(c, 0) >= 100 for c in CLASSES), f"(mínimo {min(cnt.get(c, 0) for c in CLASSES):,})")
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")
print("\nPróximo passo: me mande o texto desta saída; com as contagens eu preparo o notebook 05mc (treino do RF sem as linhas em conflito, medido na mesma validação).")

✔ A tabela de conflitos tem uma linha para cada linha da seleção do 04m (299,589)
✔ Todo conflito entre as 7 classes também é conflito entre as 13 
✔ A classe de cada linha está entre os rótulos do seu grupo de features iguais 
  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "n",\n\t\t\t"type": "UInt64"\n\t\t}\n\t],\n\n\t"data":\n\t[\n\n\t],\n\n\t"rows"); nova tentativa em 5s...
✔ A conta com as 13 classes coincide, linha a linha, com a marca conflito_rotulo do notebook 02 (0 divergências)
✔ A seleção sem conflitos = a do 04m menos as linhas conflitantes do ajuste (299,589 − 15,896 = 283,693)
✔ Nenhuma linha conflitante sobrou no ajuste 
✔ A validação ficou idêntica: mesmas linhas e mesmos pesos (59,918 linhas)
✔ Os dados sem conflitos têm as mesmas linhas da seleção sem conflitos 
✔ Dados idênticos aos originais (amostra) (14,159 linhas)
✔ prep_vw_bal_multi_treino_sem_conf: mesmas colunas da view original (46 colunas)
✔ p